# **Feature Engineering**

## **I. Import Libraries**

In [395]:
import pandas as pd
import numpy as np
import re
import os
import joblib

from pandas.api.types import is_datetime64_any_dtype, is_bool_dtype, is_numeric_dtype
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import warnings

warnings.filterwarnings("ignore")

## **II. Data Contarct and Leakage Guardrails**

In this stage, we lock the data contract for a store–department–time forecasting problem. We explicitly define the panel keys (Store, Dept, Date), verify time granularity, and construct the forecasting target using a forward shift (y at t+h). This step is essential to prevent leakage: features must be computed using only information available up to time t, while the target is taken from the future horizon (t+h). A clean contract here ensures our feature engineering and modeling pipeline is reliable, reproducible, defendable.

### 2.1 Load Dataset and Quick Schema Check

We load the cleaned dataset and perform a quick schema check to understand its structure.


In [396]:
# Import dataset
df = pd.read_parquet("data_clean/retail_fe_ready.parquet")

# Preview
print("Shape:", df.shape)
display(df.head())

print("\nColumns:")
print(df.columns.tolist())

print("\nDtypes (first 30):")
display(df.dtypes)

Shape: (421570, 16)


,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday,Type,Size
0,1,1,2010-02-05,24924.50,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False,A,151315
1,1,1,2010-02-12,46039.49,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True,A,151315
2,1,1,2010-02-19,41595.55,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False,A,151315
3,1,1,2010-02-26,19403.54,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False,A,151315
4,1,1,2010-03-05,21827.90,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False,A,151315



Columns:
['Store', 'Dept', 'Date', 'Weekly_Sales', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'IsHoliday', 'Type', 'Size']

Dtypes (first 30):


Store                    int64
Dept                     int64
Date            datetime64[ns]
Weekly_Sales           float64
Temperature            float64
Fuel_Price             float64
MarkDown1              float64
MarkDown2              float64
MarkDown3              float64
MarkDown4              float64
MarkDown5              float64
CPI                    float64
Unemployment           float64
IsHoliday                 bool
Type                    object
Size                     int64
dtype: object

### 2.2 Auto-Detect Key Columns

In this step, we explicitly define the panel keys for our forecasting problem: Store, Department, and Date.
Although the dataset already provides clear column names, making this step explicit helps prevent silent errors, improves reproducibility, and ensures that all downstream feature engineering follows a consistent data contract.


In [397]:
def guess_time_col(columns):
    # Time-like keywords
    time_kw = r"(date|datetime|timestamp|time|week|month|year)"
    # Exclude target-like keywords to avoid false positives like "weekly_sales"
    target_kw = r"(sales|revenue|units|qty|quantity|demand|volume)"
    
    candidates = []
    for c in columns:
        c_low = str(c).lower()
        if re.search(time_kw, c_low) and not re.search(target_kw, c_low):
            candidates.append(c)
    return candidates

def guess_store_col(columns):
    store_kw = r"(store|outlet|branch)"
    return [c for c in columns if re.search(store_kw, str(c), re.I)]

def guess_dim_col(columns):
    # "dim" here is the product-like dimension; in this dataset it's Dept/Department
    dim_kw = r"(dept|department|category|product|item|sku|material)"
    return [c for c in columns if re.search(dim_kw, str(c), re.I)]

time_candidates  = guess_time_col(df.columns)
store_candidates = guess_store_col(df.columns)
dim_candidates   = guess_dim_col(df.columns)

print("Time column candidates:", time_candidates)
print("Store column candidates:", store_candidates)
print("Dept/Dimension candidates:", dim_candidates)

# Auto pick with safe defaults
TIME_COL  = "Date" if "Date" in df.columns else (time_candidates[0] if time_candidates else None)
STORE_COL = "Store" if "Store" in df.columns else (store_candidates[0] if store_candidates else None)
DIM_COL   = "Dept" if "Dept" in df.columns else ("Department" if "Department" in df.columns else (dim_candidates[0] if dim_candidates else None))

print("\nSelected keys:")
print("TIME_COL  =", TIME_COL)
print("STORE_COL =", STORE_COL)
print("DIM_COL   =", DIM_COL)

assert TIME_COL is not None,  "No time-like column detected. Please set TIME_COL manually."
assert STORE_COL is not None, "No store-like column detected. Please set STORE_COL manually."
assert DIM_COL is not None,   "No department/dimension column detected. Please set DIM_COL manually."

# Ensure datetime
if not np.issubdtype(df[TIME_COL].dtype, np.datetime64):
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")

print(f"\nMissing rate in TIME_COL ({TIME_COL}): {df[TIME_COL].isna().mean():.4f}")

# Sort panel
df = df.sort_values([STORE_COL, DIM_COL, TIME_COL]).reset_index(drop=True)

# Check duplicates for (store, dept, time)
dup_rate = df.duplicated([STORE_COL, DIM_COL, TIME_COL]).mean()
print(f"Duplicate rate for panel keys (store-dept-time): {dup_rate:.6f}")

display(df[[STORE_COL, DIM_COL, TIME_COL]].head())

Time column candidates: ['Date']
Store column candidates: ['Store']
Dept/Dimension candidates: ['Dept']

Selected keys:
TIME_COL  = Date
STORE_COL = Store
DIM_COL   = Dept

Missing rate in TIME_COL (Date): 0.0000
Duplicate rate for panel keys (store-dept-time): 0.000000


,Store,Dept,Date
0,1,1,2010-02-05
1,1,1,2010-02-12
2,1,1,2010-02-19
3,1,1,2010-02-26
4,1,1,2010-03-05


### 2.3 Define Target for Weekly Sales

Here we define the forecasting target as next-week sales (t+1) for each store–department pair.
By shifting the target forward, we ensure a clear separation between features available at time t and the outcome we aim to predict, which is critical to avoid data leakage in time-series forecasting.


In [398]:
def guess_sales_col(columns):
    # Prioritize common sales (target) names; we expect Weekly_Sales / weekly_Sales
    preferred = ["Weekly_Sales", "weekly_Sales", "weekly_sales"]
    for p in preferred:
        if p in columns:
            return p
    
    # Fallback heuristic
    candidates = [c for c in columns if re.search(r"(sales|revenue|units|qty|quantity|demand|volume)", str(c), re.I)]
    return candidates[0] if candidates else None

SALES_COL = guess_sales_col(df.columns)
assert SALES_COL is not None, "No target-like column detected. Please set SALES_COL manually."

HORIZON = 1  # Predict t+1 week

Y_COL = f"y_lead_{HORIZON}"

print("SALES_COL =", SALES_COL)
print("HORIZON =", HORIZON)
print("Y_COL =", Y_COL)

# y(t+h) within each (store, dept)
df[Y_COL] = (
    df.groupby([STORE_COL, DIM_COL])[SALES_COL]
      .shift(-HORIZON)
)

# Drop rows where y is missing (end-of-series for each panel)
before = len(df)
df_model_base = df.dropna(subset=[Y_COL]).copy()
after = len(df_model_base)

print(f"\nRows before dropping missing target: {before:,}")
print(f"Rows after  dropping missing target: {after:,} (dropped {before-after:,})")

display(df_model_base[[STORE_COL, DIM_COL, TIME_COL, SALES_COL, Y_COL]].head())


SALES_COL = Weekly_Sales
HORIZON = 1
Y_COL = y_lead_1

Rows before dropping missing target: 421,570
Rows after  dropping missing target: 418,239 (dropped 3,331)


,Store,Dept,Date,Weekly_Sales,y_lead_1
0,1,1,2010-02-05,24924.50,46039.49
1,1,1,2010-02-12,46039.49,41595.55
2,1,1,2010-02-19,41595.55,19403.54
3,1,1,2010-02-26,19403.54,21827.90
4,1,1,2010-03-05,21827.90,21043.39


### 2.4 Early Leakage Scan

We perform an early name-based scan to flag potential leakage risks before feature engineering, serving as a lightweight guardrail rather than a full audit.


In [399]:
suspicious_patterns = r"(future|lead|next|t\+|label|target|y_|actual_next|after)"
potential_leak_cols = [
    c for c in df_model_base.columns
    if re.search(suspicious_patterns, str(c), re.I)
    and c not in [SALES_COL, Y_COL]
]

print("Potential leakage columns (name-based scan):")
print(potential_leak_cols if potential_leak_cols else "None detected.")


Potential leakage columns (name-based scan):
None detected.


## **III. Canonical Index and Sorting**

In this step, we formalize the dataset as a time-series panel by defining canonical panel keys (Store, Department, Date) and sorting the data accordingly. This is a critical foundation for all downstream time-based feature engineering such as lag features, rolling statistics, and trend calculations. Without a consistent panel structure and proper ordering, time-series features can silently produce incorrect results.

### 3.1 Set Canonical Panel Keys and Srot

In [400]:
# Canonical panel keys (explicit, no ambiguity)
PANEL_KEYS = [STORE_COL, DIM_COL, TIME_COL]

print("Panel keys:", PANEL_KEYS)

# Safety check: ensure no missing values in panel keys
missing_panel_rate = df[PANEL_KEYS].isna().mean()
print("\nMissing rate per panel key:")
display(missing_panel_rate)

assert missing_panel_rate.max() == 0, "Missing values detected in panel keys."

# Sort by panel keys + time (MANDATORY for lag/rolling)
df = (
    df.sort_values(PANEL_KEYS)
      .reset_index(drop=True)
)

# Final sanity check
dup_rate = df.duplicated(PANEL_KEYS).mean()
print(f"\nDuplicate rate after sorting (store-dept-time): {dup_rate:.6f}")

display(df[PANEL_KEYS].head())

Panel keys: ['Store', 'Dept', 'Date']

Missing rate per panel key:


Store    0.0
Dept     0.0
Date     0.0
dtype: float64


Duplicate rate after sorting (store-dept-time): 0.000000


,Store,Dept,Date
0,1,1,2010-02-05
1,1,1,2010-02-12
2,1,1,2010-02-19
3,1,1,2010-02-26
4,1,1,2010-03-05


### 3.2 Calendar Spine (Weekly)

In [401]:
# Infer global time range
min_date = df[TIME_COL].min()
max_date = df[TIME_COL].max()

print("Date range:", min_date, "→", max_date)

# Weekly frequency (because target is Weekly_Sales)
calendar = pd.DataFrame({
    TIME_COL: pd.date_range(start=min_date, end=max_date, freq="W-FRI")
})

print("Calendar spine preview:")
display(calendar.head())

# Get unique store–dept combinations
store_dept = df[[STORE_COL, DIM_COL]].drop_duplicates()

print("Unique store-dept pairs:", store_dept.shape[0])

# Cartesian product: (store, dept) × calendar
panel_spine = (
    store_dept.assign(key=1)
    .merge(calendar.assign(key=1), on="key")
    .drop("key", axis=1)
)

print("Panel spine shape:", panel_spine.shape)
display(panel_spine.head())

Date range: 2010-02-05 00:00:00 → 2012-10-26 00:00:00
Calendar spine preview:


,Date
0,2010-02-05
1,2010-02-12
2,2010-02-19
3,2010-02-26
4,2010-03-05


Unique store-dept pairs: 3331
Panel spine shape: (476333, 3)


,Store,Dept,Date
0,1,1,2010-02-05
1,1,1,2010-02-12
2,1,1,2010-02-19
3,1,1,2010-02-26
4,1,1,2010-03-05


### 3.3 Reindex Original Data Onto Calendar Spine

In [402]:
# Merge original data onto the full panel spine
df_panel = (
    panel_spine
    .merge(
        df,
        on=[STORE_COL, DIM_COL, TIME_COL],
        how="left"
    )
    .sort_values(PANEL_KEYS)
    .reset_index(drop=True)
)

print("Panel shape after reindexing:", df_panel.shape)

# Check how many rows were added
added_rows = len(df_panel) - len(df)
print(f"Rows added by calendar spine: {added_rows:,}")

display(df_panel[PANEL_KEYS + [SALES_COL]].head(12))


Panel shape after reindexing: (476333, 17)
Rows added by calendar spine: 54,763


,Store,Dept,Date,Weekly_Sales
0,1,1,2010-02-05,24924.50
1,1,1,2010-02-12,46039.49
2,1,1,2010-02-19,41595.55
3,1,1,2010-02-26,19403.54
4,1,1,2010-03-05,21827.90
5,1,1,2010-03-12,21043.39
6,1,1,2010-03-19,22136.64
7,1,1,2010-03-26,26229.21
8,1,1,2010-04-02,57258.43
9,1,1,2010-04-09,42960.91


**Insight**

After constructing the canonical panel and reindexing onto a complete weekly calendar spine, the dataset is transformed into a fully regularized time-series panel at the store–department level.

The increase in row count indicates that previously missing store–department–week combinations have been explicitly materialized rather than implicitly ignored. This is a deliberate design choice: missing sales are no longer hidden gaps, but observable signals that the model can learn from. It also ensures that all lagged and rolling features will be computed on a consistent temporal structure, which is critical for stability and correctness in demand forecasting.

From a business perspective, this structure aligns the dataset with real-world planning cycles, where decisions must still be made even when historical sales are absent or intermittent.

## **IV. Demans Dynamics and Seasonality (Store-Dept Weekly Panel)**

In this section, we engineer core time-series features that capture demand dynamics at the store–department level. We create lag features, rolling statistics, and simple momentum/trend indicators using only historical sales information available up to time t. We also add seasonality signals from the date and holiday flags. These features are designed to make the forecast not only accurate, but also interpretable for decision support.

### 4.1 Setup and Past-Only Guardrail

In this step, we prepare the dataset so that all engineered features strictly follow a past-only rule.

Every feature is constructed using information available up to time t, to predict demand at t + horizon.

This guardrail prevents data leakage and ensures the model behaves like a real-world forecasting system.

In [403]:
# We engineer features at time t to predict y(t+HORIZON)
# Only use info available up to time t (past-only for lags/rolling)

PANEL_KEYS = [STORE_COL, DIM_COL, TIME_COL]

# Work on a copy
df_feat = df_panel.copy()

# Ensure sorted (mandatory)
df_feat = df_feat.sort_values(PANEL_KEYS).reset_index(drop=True)

print("df_feat shape:", df_feat.shape)
display(df_feat[PANEL_KEYS + [SALES_COL]].head(10))

df_feat shape: (476333, 17)


,Store,Dept,Date,Weekly_Sales
0,1,1,2010-02-05,24924.50
1,1,1,2010-02-12,46039.49
2,1,1,2010-02-19,41595.55
3,1,1,2010-02-26,19403.54
4,1,1,2010-03-05,21827.90
5,1,1,2010-03-12,21043.39
6,1,1,2010-03-19,22136.64
7,1,1,2010-03-26,26229.21
8,1,1,2010-04-02,57258.43
9,1,1,2010-04-09,42960.91


### 4.2 Seasonality

Here, we extract calendar-based signals from the date column, such as year, month, quarter, and week-of-year. These features help the model learn recurring demand patterns that repeat over time, especially in retail.

In [404]:
# Seasonality features from Date
df_feat["year"] = df_feat[TIME_COL].dt.year
df_feat["month"] = df_feat[TIME_COL].dt.month
df_feat["quarter"] = df_feat[TIME_COL].dt.quarter

# Weekly seasonality (ISO week)
df_feat["weekofyear"] = df_feat[TIME_COL].dt.isocalendar().week.astype(int)

# Simple month boundary signal (sometimes affects retail planning)
df_feat["is_month_start"] = df_feat[TIME_COL].dt.is_month_start.astype(int)
df_feat["is_month_end"] = df_feat[TIME_COL].dt.is_month_end.astype(int)

# Ensure holiday is numeric (good for models)
if "IsHoliday" in df_feat.columns:
    df_feat["is_holiday"] = df_feat["IsHoliday"].fillna(False).astype(bool).astype(int)

display(df_feat[[TIME_COL, "year", "month", "quarter", "weekofyear", "is_month_start", "is_month_end"]].head(10))


,Date,year,month,quarter,weekofyear,is_month_start,is_month_end
0,2010-02-05,2010,2,1,5,0,0
1,2010-02-12,2010,2,1,6,0,0
2,2010-02-19,2010,2,1,7,0,0
3,2010-02-26,2010,2,1,8,0,0
4,2010-03-05,2010,3,1,9,0,0
5,2010-03-12,2010,3,1,10,0,0
6,2010-03-19,2010,3,1,11,0,0
7,2010-03-26,2010,3,1,12,0,0
8,2010-04-02,2010,4,2,13,0,0
9,2010-04-09,2010,4,2,14,0,0


### 4.3 Lag Features (Demand History per Store-Dept)

Lag features represent historical demand levels at previous time steps (e.g. last week, two weeks ago). They act as the model’s memory of recent sales behavior for each store–department pair.

In [405]:
# Lag features
LAGS = [1, 2, 4, 8]

# Grouped target series (must be sorted by PANEL_KEYS already)
g = df_feat.groupby(PANEL_KEYS[:-1])[SALES_COL]  # group by Store, Dept (time is last key)

for k in LAGS:
   df_feat[f"sales_lag_{k}"] = g.shift(k)

# Quick peek (display only)
cols_preview = PANEL_KEYS + [SALES_COL] + [f"sales_lag_{k}" for k in LAGS]
display(df_feat[cols_preview].head(12))

,Store,Dept,Date,Weekly_Sales,sales_lag_1,sales_lag_2,sales_lag_4,sales_lag_8
0,1,1,2010-02-05,24924.50,NaN,NaN,NaN,NaN
1,1,1,2010-02-12,46039.49,24924.50,NaN,NaN,NaN
2,1,1,2010-02-19,41595.55,46039.49,24924.50,NaN,NaN
3,1,1,2010-02-26,19403.54,41595.55,46039.49,NaN,NaN
4,1,1,2010-03-05,21827.90,19403.54,41595.55,24924.50,NaN
5,1,1,2010-03-12,21043.39,21827.90,19403.54,46039.49,NaN
6,1,1,2010-03-19,22136.64,21043.39,21827.90,41595.55,NaN
7,1,1,2010-03-26,26229.21,22136.64,21043.39,19403.54,NaN
8,1,1,2010-04-02,57258.43,26229.21,22136.64,21827.90,24924.50
9,1,1,2010-04-09,42960.91,57258.43,26229.21,21043.39,46039.49


In [406]:
# Sanity check (meaningful)
assert not any(c.endswith("_lag_0") for c in df_feat.columns), "Found lag_0 column => leakage risk."

lag_cols = [f"sales_lag_{k}" for k in LAGS]
print("Negative values in lag cols are allowed in this dataset.")
print("Negative count:", int((df_feat[lag_cols] < 0).sum().sum()))

# If you kept log_lags, check only for +/- inf (NaN from shift is expected before imputation)
log_cols = [c for c in df_feat.columns if c.startswith("log_sales_lag_")]
if len(log_cols) > 0:
    inf_rows = np.isinf(df_feat[log_cols].to_numpy()).any(axis=1).sum()
    print("Rows with INF in log lag cols:", int(inf_rows))


Negative values in lag cols are allowed in this dataset.
Negative count: 5025


### 4.4 Rolling Statistics (Stability and Volatility)

Rolling statistics summarize demand behavior over a window of time, such as average, volatility, minimum, and maximum sales. They help capture whether demand is stable, fluctuating, or volatile.

This step matters because two products with the same average sales can require very different decisions if one is volatile and the other is stable.

In [407]:
# Rolling statistics
ROLL_WINDOWS = [4, 8]

# Use past-only series (exclude current week)
past_sales = g.shift(1)

for w in ROLL_WINDOWS:
    df_feat[f"roll_mean_{w}"] = past_sales.rolling(window=w, min_periods=1).mean()
    df_feat[f"roll_std_{w}"]  = past_sales.rolling(window=w, min_periods=1).std()
    df_feat[f"roll_min_{w}"]  = past_sales.rolling(window=w, min_periods=1).min()
    df_feat[f"roll_max_{w}"]  = past_sales.rolling(window=w, min_periods=1).max()

display(df_feat[PANEL_KEYS + [f"roll_mean_{w}" for w in ROLL_WINDOWS] + [f"roll_std_{w}" for w in ROLL_WINDOWS]].head(12))


,Store,Dept,Date,roll_mean_4,roll_mean_8,roll_std_4,roll_std_8
0,1,1,2010-02-05,NaN,NaN,NaN,NaN
1,1,1,2010-02-12,24924.500000,24924.500000,NaN,NaN
2,1,1,2010-02-19,35481.995000,35481.995000,14930.552614,14930.552614
3,1,1,2010-02-26,37519.846667,37519.846667,11131.900957,11131.900957
4,1,1,2010-03-05,32990.770000,32990.770000,12832.106391,12832.106391
5,1,1,2010-03-12,32216.620000,30758.196000,13554.047185,12182.739805
6,1,1,2010-03-19,25967.595000,29139.061667,10467.484020,11595.899933
7,1,1,2010-03-26,21102.867500,28138.715714,1222.784968,10911.412537
8,1,1,2010-04-02,22809.285000,27900.027500,2325.929203,10124.538627
9,1,1,2010-04-09,31666.917500,31941.768750,17206.391261,14342.348043


### 4.5 Trend/Momentum

Trend and momentum features measure directional change, such as week-over-week growth or deviation from recent averages. They indicate whether demand is accelerating, slowing down, or reverting to normal.

In [408]:
eps = 1e-6

# Week-over-week momentum using only past lags
df_feat["mom_wow"] = (df_feat["sales_lag_1"] - df_feat["sales_lag_2"]) / (np.abs(df_feat["sales_lag_2"]) + eps)

# week momentum proxy (lag_1 vs lag_4)
df_feat["mom_4w"] = (df_feat["sales_lag_1"] - df_feat["sales_lag_4"]) / (np.abs(df_feat["sales_lag_4"]) + eps)

# Deviation from recent baseline using latest observed (lag_1) vs rolling mean (already past-only)
if "roll_mean_4" in df_feat.columns:
    df_feat["dev_from_roll4"] = (df_feat["sales_lag_1"] - df_feat["roll_mean_4"]) / (np.abs(df_feat["roll_mean_4"]) + eps)
if "roll_mean_8" in df_feat.columns:
    df_feat["dev_from_roll8"] = (df_feat["sales_lag_1"] - df_feat["roll_mean_8"]) / (np.abs(df_feat["roll_mean_8"]) + eps)

# Quick peek (fixed column list, no weird strings)
show_cols = PANEL_KEYS + [
    SALES_COL,
    "sales_lag_1", "sales_lag_2", "sales_lag_4",
]
for c in ["roll_mean_4", "mom_wow", "mom_4w", "dev_from_roll4"]:
    if c in df_feat.columns:
        show_cols.append(c)

display(df_feat[show_cols].head(12))

,Store,Dept,Date,Weekly_Sales,sales_lag_1,sales_lag_2,sales_lag_4,roll_mean_4,mom_wow,mom_4w,dev_from_roll4
0,1,1,2010-02-05,24924.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,2010-02-12,46039.49,24924.50,NaN,NaN,24924.500000,NaN,NaN,0.000000
2,1,1,2010-02-19,41595.55,46039.49,24924.50,NaN,35481.995000,0.847158,NaN,0.297545
3,1,1,2010-02-26,19403.54,41595.55,46039.49,NaN,37519.846667,-0.096525,NaN,0.108628
4,1,1,2010-03-05,21827.90,19403.54,41595.55,24924.50,32990.770000,-0.533519,-0.221507,-0.411849
5,1,1,2010-03-12,21043.39,21827.90,19403.54,46039.49,32216.620000,0.124944,-0.525887,-0.322465
6,1,1,2010-03-19,22136.64,21043.39,21827.90,41595.55,25967.595000,-0.035941,-0.494095,-0.189629
7,1,1,2010-03-26,26229.21,22136.64,21043.39,19403.54,21102.867500,0.051952,0.140856,0.048987
8,1,1,2010-04-02,57258.43,26229.21,22136.64,21827.90,22809.285000,0.184878,0.201637,0.149936
9,1,1,2010-04-09,42960.91,57258.43,26229.21,21043.39,31666.917500,1.183002,1.720970,0.808147


### 4.6 Quick Sanity Check

In [409]:
feature_cols_created = [c for c in df_feat.columns if c.startswith(("sales_lag_", "log_sales_lag_", "roll_", "mom_", "dev_"))]
print("Number of engineered features:", len(feature_cols_created))
print("Sample engineered features:", feature_cols_created[:15])

missing_rate = df_feat[feature_cols_created].isna().mean().sort_values(ascending=False).head(15)
print("\nTop missing rates (expected at series start):")
display(missing_rate)


Number of engineered features: 16
Sample engineered features: ['sales_lag_1', 'sales_lag_2', 'sales_lag_4', 'sales_lag_8', 'roll_mean_4', 'roll_std_4', 'roll_min_4', 'roll_max_4', 'roll_mean_8', 'roll_std_8', 'roll_min_8', 'roll_max_8', 'mom_wow', 'mom_4w', 'dev_from_roll4']

Top missing rates (expected at series start):


sales_lag_8       0.164784
mom_4w            0.153966
sales_lag_4       0.139898
mom_wow           0.139535
sales_lag_2       0.127373
sales_lag_1       0.121180
dev_from_roll4    0.121180
dev_from_roll8    0.121180
roll_std_4        0.109430
roll_std_8        0.092110
roll_mean_4       0.089053
roll_min_4        0.089053
roll_max_4        0.089053
roll_mean_8       0.073680
roll_min_8        0.073680
dtype: float64

**Summary**

Together, these features convert raw weekly sales into structured demand signals that support forecasting, ranking, and decision-making.

## **V. Cross Sectional Context**

In this section, we add cross-sectional context features—signals that compare a store-dept’s demand against (1) the store’s overall baseline and (2) the product/department’s baseline across stores. This is important because forecasting isn’t only about “what happened last week,” but also about “how this series performs relative to its peers.” We compute these features using past-only components (e.g., lag/rolling that exclude the current week) so the model can learn ranking and decision-friendly signals without leakage.

### 5.1 Define Part-Only Base Signal

We define a single, stable base demand proxy that represents recent demand without using current-week information. This base signal becomes the foundation for all cross-sectional comparisons to ensure consistency and leakage-free feature construction.

In [410]:
eps = 1e-6

# We'll use past-only signals for cross-sectional context
# Prefer rolling mean that excludes current week (we built roll_mean_* from shifted series)
BASE_DEMAND_COL = "roll_mean_4" if "roll_mean_4" in df_feat.columns else "sales_lag_1"

assert BASE_DEMAND_COL in df_feat.columns, f"{BASE_DEMAND_COL} not found. Check your lag/rolling features."

print("BASE_DEMAND_COL =", BASE_DEMAND_COL)
display(df_feat[PANEL_KEYS + [SALES_COL, BASE_DEMAND_COL]].head())

BASE_DEMAND_COL = roll_mean_4


,Store,Dept,Date,Weekly_Sales,roll_mean_4
0,1,1,2010-02-05,24924.50,NaN
1,1,1,2010-02-12,46039.49,24924.500000
2,1,1,2010-02-19,41595.55,35481.995000
3,1,1,2010-02-26,19403.54,37519.846667
4,1,1,2010-03-05,21827.90,32990.770000


### 5.2 Store Context + Product Context

This step builds contextual baselines at two levels:
- Store context: how demand is distributed across departments within the same store.
- Product (dept) context: how a department performs across different stores.

We also normalize volatility to capture stability vs. risk, which is important for operational planning.

In [411]:
# Store context (within a store, across depts) using past-only demand proxy
store_ctx = (
    df_feat.groupby([STORE_COL, TIME_COL], as_index=False)[BASE_DEMAND_COL]
    .agg(store_baseline_mean="mean", store_baseline_std="std")
)

# PRODUCT/DEPT context (within a dept, across stores) using past-only demand proxy
dept_ctx = (
    df_feat.groupby([DIM_COL, TIME_COL], as_index=False)[BASE_DEMAND_COL]
    .agg(dept_baseline_mean="mean", dept_baseline_std="std")
)

# Merge contexts back
df_feat = (
    df_feat
    .merge(store_ctx, on=[STORE_COL, TIME_COL], how="left")
    .merge(dept_ctx, on=[DIM_COL, TIME_COL], how="left")
)

# Stability ratios (volatility normalized)
df_feat["store_cv"] = df_feat["store_baseline_std"] / (df_feat["store_baseline_mean"] + eps)
df_feat["dept_cv"]  = df_feat["dept_baseline_std"]  / (df_feat["dept_baseline_mean"]  + eps)

display(df_feat[PANEL_KEYS + [BASE_DEMAND_COL, "store_baseline_mean", "dept_baseline_mean", "store_cv", "dept_cv"]].head(12))


,Store,Dept,Date,roll_mean_4,store_baseline_mean,dept_baseline_mean,store_cv,dept_cv
0,1,1,2010-02-05,NaN,21996.451343,532.676481,1.334097,2.043289
1,1,1,2010-02-12,24924.500000,22073.290241,14503.133481,1.190063,0.715162
2,1,1,2010-02-19,35481.995000,22784.326579,25389.252185,1.175291,0.549980
3,1,1,2010-02-26,37519.846667,22361.735205,25609.430889,1.248545,0.522821
4,1,1,2010-03-05,32990.770000,21597.777740,22992.581944,1.259023,0.518634
5,1,1,2010-03-12,32216.620000,20999.685405,22330.466167,1.276987,0.510071
6,1,1,2010-03-19,25967.595000,20319.321047,18701.046889,1.294225,0.519544
7,1,1,2010-03-26,21102.867500,19846.397466,17192.319722,1.302955,0.540314
8,1,1,2010-04-02,22809.285000,19828.499932,19216.391056,1.301922,0.567225
9,1,1,2010-04-09,31666.917500,20237.539658,28601.390889,1.269490,0.636550


### 5.3 Relative Features

Here we convert absolute demand signals into relative and ranking-friendly features, answering questions like:
- Is this department over- or under-performing relative to its store?
- How strong is it compared to the same department in other stores?
- What is its contribution to total store demand?

These features enable downstream ranking, prioritization, and decision rules, not just point forecasting.

In [412]:
# Relative to store baseline and dept baseline (ranking-friendly signals)
df_feat["demand_vs_store_baseline"]  = df_feat[BASE_DEMAND_COL] / (df_feat["store_baseline_mean"] + eps)
df_feat["demand_vs_dept_baseline"]   = df_feat[BASE_DEMAND_COL] / (df_feat["dept_baseline_mean"] + eps)

# Percentile rank within store (per date): "where does this dept stand inside the store?"
df_feat["pct_rank_in_store"] = (
    df_feat.groupby([STORE_COL, TIME_COL])[BASE_DEMAND_COL]
    .rank(pct=True, method="average")
)

# Share of store demand (past-only proxy): dept contribution to store baseline
store_sum = df_feat.groupby([STORE_COL, TIME_COL])[BASE_DEMAND_COL].transform("sum")
df_feat["share_of_store_demand"] = df_feat[BASE_DEMAND_COL] / (store_sum + eps)

cols_preview = PANEL_KEYS + [
    BASE_DEMAND_COL,
    "store_baseline_mean", "dept_baseline_mean",
    "demand_vs_store_baseline", "demand_vs_dept_baseline",
    "pct_rank_in_store", "share_of_store_demand"
]

display(df_feat[cols_preview].head(10))


,Store,Dept,Date,roll_mean_4,store_baseline_mean,dept_baseline_mean,demand_vs_store_baseline,demand_vs_dept_baseline,pct_rank_in_store,share_of_store_demand
0,1,1,2010-02-05,NaN,21996.451343,532.676481,NaN,NaN,NaN,NaN
1,1,1,2010-02-12,24924.500000,22073.290241,14503.133481,1.129170,1.718560,0.671053,0.014858
2,1,1,2010-02-19,35481.995000,22784.326579,25389.252185,1.557298,1.397520,0.842105,0.020491
3,1,1,2010-02-26,37519.846667,22361.735205,25609.430889,1.677859,1.465079,0.821918,0.022984
4,1,1,2010-03-05,32990.770000,21597.777740,22992.581944,1.527508,1.434844,0.794521,0.020925
5,1,1,2010-03-12,32216.620000,20999.685405,22330.466167,1.534148,1.442720,0.797297,0.020732
6,1,1,2010-03-19,25967.595000,20319.321047,18701.046889,1.277976,1.388564,0.756757,0.017270
7,1,1,2010-03-26,21102.867500,19846.397466,17192.319722,1.063310,1.227459,0.716216,0.014369
8,1,1,2010-04-02,22809.285000,19828.499932,19216.391056,1.150328,1.186970,0.716216,0.015545
9,1,1,2010-04-09,31666.917500,20237.539658,28601.390889,1.564761,1.107181,0.794521,0.021435


## **VI. Freeze Feature List**

In this step, we “freeze” the final modeling feature set programmatically from the engineered dataframe. We exclude panel keys (Store/Dept/Date) and the target column to avoid hardcoded lists and prevent silent bugs when columns change. We also split features into numeric vs categorical to make the next preprocessing/pipeline step clean and reproducible.

In [413]:
# Required guardrails
assert "df_feat" in globals(), "df_feat not found. Run feature engineering steps first."
assert "PANEL_KEYS" in globals(), "PANEL_KEYS not found. Define panel keys first."
assert "SALES_COL" in globals(), "SALES_COL not found. Define target column first."

for k in PANEL_KEYS:
    assert k in df_feat.columns, f"Panel key '{k}' not found in df_feat."
assert SALES_COL in df_feat.columns, f"Target '{SALES_COL}' not found in df_feat."

# Exclude columns from training features
EXCLUDE_FROM_FEATURES = set(PANEL_KEYS + [SALES_COL])

# Exclude truly-raw / non-feature columns if they exist (keep minimal & safe)
RAW_EXCLUDE = set([]) 
EXCLUDE_FROM_FEATURES |= RAW_EXCLUDE

# Build feature list programmatically
FEATURE_COLS = [c for c in df_feat.columns if c not in EXCLUDE_FROM_FEATURES]

# Split numeric vs categorical for preprocessing
NUMERIC_FEATURES = []
CATEGORICAL_FEATURES = []

for c in FEATURE_COLS:
    # Skip datetime columns if any accidentally slip in
    if is_datetime64_any_dtype(df_feat[c]):
        continue

    # Treat bool as numeric (0/1) for most models
    if is_bool_dtype(df_feat[c]):
        NUMERIC_FEATURES.append(c)
    elif is_numeric_dtype(df_feat[c]):
        NUMERIC_FEATURES.append(c)
    else:
        CATEGORICAL_FEATURES.append(c)

# Low-card vs high-card categoricals (for later preprocessing decisions)
HIGH_CARD_THRESHOLD = 50
if len(CATEGORICAL_FEATURES) > 0:
    cat_nunique = df_feat[CATEGORICAL_FEATURES].nunique(dropna=True)
else:
    cat_nunique = pd.Series(dtype=int)

CATEGORICAL_LOW_CARD = cat_nunique[cat_nunique <= HIGH_CARD_THRESHOLD].index.tolist()
CATEGORICAL_HIGH_CARD = cat_nunique[cat_nunique > HIGH_CARD_THRESHOLD].index.tolist()

# Final exports (consistent naming for later steps)
FEATURES = FEATURE_COLS  # alias for readability in later cells

print("== Freeze Feature List ==")
print("Target                   :", SALES_COL)
print("Panel keys               :", PANEL_KEYS)
print("Total columns in df_feat :", df_feat.shape[1])
print("Feature columns          :", len(FEATURES))
print(" - Numeric               :", len(NUMERIC_FEATURES))
print(" - Categorical           :", len(CATEGORICAL_FEATURES))
print("   - Low-card            :", len(CATEGORICAL_LOW_CARD))
print("   - High-card           :", len(CATEGORICAL_HIGH_CARD))

print("\nSample numeric features     :", NUMERIC_FEATURES[:15])
print("Sample categorical features :", CATEGORICAL_FEATURES[:15])


== Freeze Feature List ==
Target                   : Weekly_Sales
Panel keys               : ['Store', 'Dept', 'Date']
Total columns in df_feat : 50
Feature columns          : 46
 - Numeric               : 44
 - Categorical           : 2
   - Low-card            : 2
   - High-card           : 0

Sample numeric features     : ['Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Size', 'y_lead_1', 'year', 'month', 'quarter', 'weekofyear']
Sample categorical features : ['IsHoliday', 'Type']


## **VII. Missing Values and Outliers Handling**

Here we handle missing values in a time-aware way. Lags/rolling/context naturally create NaNs at the beginning of each series, so we treat them safely without using future information. We also avoid global outlier capping before the time split to prevent subtle time leakage; if needed, we will fit caps on the training period only later.

### 7.1 Quick Audit Missingness

In [414]:
# Quick Audit Missingness (use frozen FEATURES)
treat_cols = [c for c in FEATURES if c in df_feat.columns]

miss_sales = df_feat[SALES_COL].isna().mean()
miss_features = df_feat[treat_cols].isna().mean().sort_values(ascending=False)

print(f"Missing rate (SALES_COL={SALES_COL}): {miss_sales:.4f}")
print("\nTop missing rates (FEATURES):")
display(miss_features.head(15))

Missing rate (SALES_COL=Weekly_Sales): 0.1150

Top missing rates (FEATURES):


MarkDown2         0.766449
MarkDown4         0.716654
MarkDown3         0.712195
MarkDown1         0.683665
MarkDown5         0.682088
sales_lag_8       0.164784
mom_4w            0.153966
sales_lag_4       0.139898
mom_wow           0.139535
sales_lag_2       0.127373
y_lead_1          0.121961
sales_lag_1       0.121180
dev_from_roll8    0.121180
dev_from_roll4    0.121180
Fuel_Price        0.114968
dtype: float64

### 7.2 Safe Missing Handling Rules

In [415]:
treat_cols = [c for c in FEATURES if c in df_feat.columns]

# Only add missing flags for columns that actually have missing values
cols_with_missing = [c for c in treat_cols if df_feat[c].isna().any()]

for c in cols_with_missing:
    df_feat[f"{c}_missing"] = df_feat[c].isna().astype(int)

# Fill numeric NaNs with 0 (safe for lag/rolling/context cold-start)
num_cols = [c for c in treat_cols if is_numeric_dtype(df_feat[c]) or is_bool_dtype(df_feat[c])]
df_feat[num_cols] = df_feat[num_cols].fillna(0)

# Fill categorical NaNs with "Unknown"
cat_cols = [c for c in treat_cols if c not in num_cols]
for c in cat_cols:
    df_feat[c] = df_feat[c].fillna("Unknown")

print("Done. Example preview:")
preview_cols = PANEL_KEYS + [SALES_COL] + treat_cols[:5]
# add 1 missing-flag column if exists
if len(cols_with_missing) > 0:
    preview_cols += [f"{cols_with_missing[0]}_missing"]
display(df_feat[preview_cols].head(10))


Done. Example preview:


,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,Temperature_missing
0,1,1,2010-02-05,24924.50,42.31,2.572,0.0,0.0,0.0,0
1,1,1,2010-02-12,46039.49,38.51,2.548,0.0,0.0,0.0,0
2,1,1,2010-02-19,41595.55,39.93,2.514,0.0,0.0,0.0,0
3,1,1,2010-02-26,19403.54,46.63,2.561,0.0,0.0,0.0,0
4,1,1,2010-03-05,21827.90,46.50,2.625,0.0,0.0,0.0,0
5,1,1,2010-03-12,21043.39,57.79,2.667,0.0,0.0,0.0,0
6,1,1,2010-03-19,22136.64,54.58,2.720,0.0,0.0,0.0,0
7,1,1,2010-03-26,26229.21,51.45,2.732,0.0,0.0,0.0,0
8,1,1,2010-04-02,57258.43,62.27,2.719,0.0,0.0,0.0,0
9,1,1,2010-04-09,42960.91,65.86,2.770,0.0,0.0,0.0,0


### 7.3 Outliers Policy

We intentionally do not perform global outlier capping before the time-based split to avoid subtle time leakage. If we want capping, we will fit cap thresholds on the training period only and then apply them to validation/test.

In [416]:
DO_TARGET_CAPPING = False

def fit_cap_thresholds(y_train: pd.Series, q_low=0.001, q_high=0.999):
    low_cap = y_train.quantile(q_low)
    high_cap = y_train.quantile(q_high)
    return low_cap, high_cap

def apply_capping(y: pd.Series, low_cap, high_cap):
    return y.clip(lower=low_cap, upper=high_cap)

print("Outlier policy: DO_TARGET_CAPPING =", DO_TARGET_CAPPING)
print("Note: If enabled later, caps must be fitted on TRAIN split only.")


Outlier policy: DO_TARGET_CAPPING = False
Note: If enabled later, caps must be fitted on TRAIN split only.


### 7.4 Final Sanity Check

In [417]:
post_miss_features = df_feat[treat_cols].isna().mean().sort_values(ascending=False)

print("Post-imputation: Top missing rates (FEATURES):")
display(post_miss_features.head(10))

print("\nAny remaining NaN in features?", (df_feat[treat_cols].isna().sum().sum() > 0))


Post-imputation: Top missing rates (FEATURES):


Temperature       0.0
dev_from_roll4    0.0
roll_std_4        0.0
roll_min_4        0.0
roll_max_4        0.0
roll_mean_8       0.0
roll_std_8        0.0
roll_min_8        0.0
roll_max_8        0.0
mom_wow           0.0
dtype: float64


Any remaining NaN in features? False


## **VIII. Time-Based Train-Val-Test Split**

### Time-based Split Strategy (Panel Time Series)

In this project, we must evaluate the model using time-based splits, not random splits, because future weeks must never leak into the training data. We split the dataset into Train / Validation / Test by date to simulate a real forecasting workflow: train on historical weeks, validate on the most recent past window for tuning, and test on the final unseen weeks for a fair performance estimate.

In [418]:
# Config
DATE_COL = "Date"
TARGET_COL = "y_lead_1"
KEY_COLS = ["Store", "Dept"]

# Window split
TEST_WEEKS = 4
VAL_WEEKS = 8

# Sort to enforce time integrity
df_split = df_feat.sort_values(KEY_COLS + [DATE_COL]).reset_index(drop=True)

# Pick cutoff dates (global cutoffs across the full dataset timeline)
all_dates = pd.Series(df_split[DATE_COL].dropna().unique()).sort_values()
last_date = all_dates.max()

test_start = last_date - pd.Timedelta(weeks=TEST_WEEKS - 1)
val_start  = test_start - pd.Timedelta(weeks=VAL_WEEKS)

# Assign split label by date
df_split["split"] = "train"
df_split.loc[df_split[DATE_COL] >= val_start, "split"] = "val"
df_split.loc[df_split[DATE_COL] >= test_start, "split"] = "test"

# Quick sanity checks
print("Date range (min -> max):", df_split[DATE_COL].min(), "->", df_split[DATE_COL].max())
print("Cutoffs:")
print("  val_start :", val_start.date())
print("  test_start:", test_start.date())

display(df_split["split"].value_counts())

# Check split coverage per store-dept (should have all three for most panels)
coverage = (
    df_split.groupby(KEY_COLS)["split"]
    .nunique()
    .value_counts()
    .sort_index()
)
print("\n#panels by number of splits present:")
display(coverage)

Date range (min -> max): 2010-02-05 00:00:00 -> 2012-10-26 00:00:00
Cutoffs:
  val_start : 2012-08-10
  test_start: 2012-10-05


split
train    436361
val       26648
test      13324
Name: count, dtype: int64


#panels by number of splits present:


split
3    3331
Name: count, dtype: int64

### Create Train/Validation/Test Matrices (X, y) from Time-based Splits

Now that we have a leakage-safe, time-based split, we build X_train, y_train, X_val, y_val, X_test, y_test. We keep identifiers (Store, Dept, Date) separate for tracking and later decision-layer outputs, while X contains only model features and y contains the future demand target.

In [419]:
# Required columns
DATE_COL = "Date"
TARGET_COL = "y_lead_1"
KEY_COLS = ["Store", "Dept"]
SPLIT_COL = "split"

# Define feature columns (exclude identifiers + target + split)
exclude_cols = set(KEY_COLS + [DATE_COL, TARGET_COL, SPLIT_COL])
feature_cols = [c for c in df_split.columns if c not in exclude_cols]

print("Total features:", len(feature_cols))
print("Sample features:", feature_cols[:10])

# Helper to slice by split
def make_xy(df: pd.DataFrame, split_name: str):
    d = df.loc[df[SPLIT_COL] == split_name].copy()
    X = d[feature_cols].copy()
    y = d[TARGET_COL].copy()
    meta = d[KEY_COLS + [DATE_COL]].copy()  # for tracking / decision layer later
    return X, y, meta

X_train, y_train, meta_train = make_xy(df_split, "train")
X_val,   y_val,   meta_val   = make_xy(df_split, "val")
X_test,  y_test,  meta_test  = make_xy(df_split, "test")

# Sanity checks
print("\nShapes:")
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "| y_val  :", y_val.shape)
print("X_test :", X_test.shape,  "| y_test :", y_test.shape)

print("\nTarget missingness by split:")
print("train:", y_train.isna().mean())
print("val  :", y_val.isna().mean())
print("test :", y_test.isna().mean())

print("\nAny NA in features by split (should be expected for lag/rolling):")
print("train:", X_train.isna().any().any())
print("val  :", X_val.isna().any().any())
print("test :", X_test.isna().any().any())

# Quick peek at meta date ranges per split
print("\nMeta date ranges:")
print("train:", meta_train[DATE_COL].min(), "->", meta_train[DATE_COL].max())
print("val  :", meta_val[DATE_COL].min(),   "->", meta_val[DATE_COL].max())
print("test :", meta_test[DATE_COL].min(),  "->", meta_test[DATE_COL].max())

Total features: 82
Sample features: ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment']

Shapes:
X_train: (436361, 82) | y_train: (436361,)
X_val  : (26648, 82) | y_val  : (26648,)
X_test : (13324, 82) | y_test : (13324,)

Target missingness by split:
train: 0.0
val  : 0.0
test : 0.0

Any NA in features by split (should be expected for lag/rolling):
train: True
val  : True
test : True

Meta date ranges:
train: 2010-02-05 00:00:00 -> 2012-08-03 00:00:00
val  : 2012-08-10 00:00:00 -> 2012-09-28 00:00:00
test : 2012-10-05 00:00:00 -> 2012-10-26 00:00:00


## **IX. Preprocessing by Feature**

Before modeling, we remove rows where the forecasting target is missing (y_lead_1 is NaN). This happens naturally at the end of the time series (no future week available), and removing those rows is necessary for valid training and evaluation. Next, we build a preprocessing pipeline based on feature types: numeric features get imputed and (optionally) scaled, while categorical features are encoded. We fit all preprocessors on the training split only to avoid leakage.

In [420]:
# Config
TARGET_COL = "y_lead_1"

# Fnction to drop rows with missing target (per split)
def drop_missing_target(X, y, meta=None):
    mask = ~y.isna()
    X2 = X.loc[mask].copy()
    y2 = y.loc[mask].copy()
    meta2 = meta.loc[mask].copy() if meta is not None else None
    return X2, y2, meta2

X_train2, y_train2, meta_train2 = drop_missing_target(X_train, y_train, meta_train)
X_val2,   y_val2,   meta_val2   = drop_missing_target(X_val, y_val, meta_val)
X_test2,  y_test2,  meta_test2  = drop_missing_target(X_test, y_test, meta_test)

print("After dropping NaN target:")
print("train:", X_train2.shape, y_train2.shape)
print("val  :", X_val2.shape,   y_val2.shape)
print("test :", X_test2.shape,  y_test2.shape)

After dropping NaN target:
train: (436361, 82) (436361,)
val  : (26648, 82) (26648,)
test : (13324, 82) (13324,)


Before building the preprocessing pipeline, we explicitly standardize column data types to avoid silent failures and ensure robustness in production-like settings.

In real-world datasets, categorical indicators such as IsHoliday often come from multiple sources and may appear in mixed formats (e.g. True, False, "True", "False", or "Unknown"). While these values may look semantically similar, machine learning encoders (such as OneHotEncoder) require inputs to be type-consistent. Mixed boolean and string values can cause unexpected errors during fitting or transformation.

By converting IsHoliday into a numeric 0/1 representation and forcing all remaining categorical columns to a uniform string type, we ensure that:
- the preprocessing pipeline is stable and reproducible,
- encoders never encounter mixed data types,
- and the same logic can safely be applied to validation, test, and future unseen data.

This step reflects a common industry practice where data is made explicitly model-safe before entering automated pipelines.

In [421]:
# Standardize column types

# Make IsHoliday numeric 0/1 (robust to 'True'/'False' strings)
def to_bool01(s):
    return (
        s.astype("string")
         .str.lower()
         .map({"true": 1, "false": 0})
         .fillna(pd.to_numeric(s, errors="coerce"))
         .fillna(0)
         .astype("int8")
    )

for X in [X_train2, X_val2, X_test2]:
    if "IsHoliday" in X.columns:
        X["IsHoliday"] = to_bool01(X["IsHoliday"])

# Force all object/category cols to string (so OHE never sees mixed types)
obj_cols = X_train2.select_dtypes(include=["object", "category"]).columns
for X in [X_train2, X_val2, X_test2]:
    for c in obj_cols:
        X[c] = X[c].astype("string").fillna("missing")


In [422]:
# Define feature groups
# Identify columns by dtype from X_train2 (train only, to avoid peeking)
numeric_features = X_train2.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train2.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()

print("\nNumeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Categorical sample:", categorical_features[:10])


Numeric features: 81
Categorical features: 1
Categorical sample: ['Type']


In [423]:
# Build preprocessors
# Numeric: impute + scale
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())   # untuk linear models; untuk tree bisa di-drop nanti
])

# Categorical: impute + one-hot
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

preprocess

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Weekly_Sales', 'Temperature', 'Fuel_Price',
                                  'MarkDown1', 'MarkDown2', 'MarkDown3',
                                  'MarkDown4', 'MarkDown5', 'CPI',
                                  'Unemployment', 'IsHoliday', 'Size', 'year',
                                  'month', 'quarter', 'weekofyear',
                                  'is_month_start', 'is_month_end',
                                  'is_holiday', 'sales_lag_1', 'sales_lag_2',
                                  'sales_lag_4', 'sales_lag_8', 'roll_mean_4',
                                  'roll_std_4', 'roll_min_4', 'roll_max_4',
                                  'roll_mean_8', 'roll_std_8', 'roll_min_8', ...]),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Type'])])

In [424]:
# Preprocess fit and transform

# Fit on train onlt
preprocess.fit(X_train2)

# Transform
Xtr = preprocess.transform(X_train2)
Xva = preprocess.transform(X_val2)
Xte = preprocess.transform(X_test2)

print(Xtr.shape, Xva.shape, Xte.shape)

(436361, 85) (26648, 85) (13324, 85)


## **X. Save Artifacts**

After finalizing feature engineering and preprocessing, we persist several key artifacts that will be reused in the modeling stage.

This separation is intentional. In a real-world machine learning workflow, feature engineering, preprocessing, and modeling are treated as distinct stages, each with clear inputs and outputs.

By saving these artifacts:
- We ensure the exact same preprocessing logic is reused during training, validation, and testing.
- We eliminate the risk of silent inconsistencies, such as different scaling, encoding, or feature selection across notebooks.
- We make the workflow reproducible, auditable, and easier to maintain, which closely mirrors production and MLOps practices.

The artifacts saved here are:
- preprocess.joblib: the full preprocessing pipeline (imputation, scaling, encoding).
- numeric_features.joblib: the definitive list of numeric features used by the pipeline.
- categorical_features.joblib: the definitive list of categorical features used by the pipeline.

Persisting these files allows the modeling notebook to stay clean and focused purely on modeling logic, without re-implementing feature preparation steps.

In [425]:
# Create folder directory so save file
os.makedirs("artifacts", exist_ok=True)


In [426]:
# Save artifacts
joblib.dump(preprocess, "artifacts/preprocess.joblib")
joblib.dump(numeric_features, "artifacts/numeric_features.joblib")
joblib.dump(categorical_features, "artifacts/categorical_features.joblib")


['artifacts/categorical_features.joblib']

Not all artifacts are saved using joblib, and this is intentional.

Some components are persisted as Python scripts or JSON files because they represent logic and configuration, not trained objects.
- Python modules (feature_engineering.py, time_split.py), contain:
    - how features are constructed,
    - how time-based splits are defined.
- JSON files (fe_config.json, split_cutoffs.json), that store explicit configuration such as feature definitions and time boundaries for train/validation/test splits.

This mirrors real production systems, where code defines behavior, and configuration defines decisions.

---

Before moving to the modeling notebook, we perform a final consistency check to ensure all saved artifacts are valid and aligned with the training data.

This check verifies that:
- there are no duplicated feature names,
- numeric and categorical features do not overlap,
- all declared features exist in X_train,
- data types match expectations,
- and the preprocessing pipeline can successfully fit and transform train, validation, and test sets with consistent output shapes.

Passing this checkpoint means:
- the feature engineering stage is closed and stable,
- the modelling notebook can safely treat preprocessing as a black box,
- and any future results are attributable to modeling choices, not data preparation issues.

At this point, the system is ready to move forward to modeling with confidence.

In [429]:
# Final check
PREP = "artifacts/preprocess.joblib"
NUM  = "artifacts/numeric_features.joblib"
CAT  = "artifacts/categorical_features.joblib"

preprocess = joblib.load(PREP)
numeric_features = joblib.load(NUM)
categorical_features = joblib.load(CAT)

print("Loaded:")
print(" - preprocess:", type(preprocess))
print(" - #numeric:", len(numeric_features))
print(" - #categorical:", len(categorical_features))
print(" - categorical sample:", categorical_features[:10])

# 1) sanity: no duplicates
assert len(numeric_features) == len(set(numeric_features)), "Duplicate in numeric_features!"
assert len(categorical_features) == len(set(categorical_features)), "Duplicate in categorical_features!"
assert set(numeric_features).isdisjoint(set(categorical_features)), "Overlap numeric & categorical!"

# 2) sanity: columns exist in X_train2
missing_in_train = [c for c in (numeric_features + categorical_features) if c not in X_train2.columns]
assert len(missing_in_train) == 0, f"These features not found in X_train2: {missing_in_train}"

# 3) quick dtype check
print("\nDtypes snapshot:")
print(X_train2[numeric_features].dtypes.value_counts().head(10))
print(X_train2[categorical_features].dtypes.value_counts().head(10))

# 4) fit/transform check (must work)
preprocess.fit(X_train2)
Xtr = preprocess.transform(X_train2)
Xva = preprocess.transform(X_val2)
Xte = preprocess.transform(X_test2)

print("\nTransformed shapes:", Xtr.shape, Xva.shape, Xte.shape)
assert Xtr.shape[1] == Xva.shape[1] == Xte.shape[1], "Column count mismatch after transform!"
print("✅ Artifacts is consistent.")


Loaded:
 - preprocess: <class 'sklearn.compose._column_transformer.ColumnTransformer'>
 - #numeric: 81
 - #categorical: 1
 - categorical sample: ['Type']

Dtypes snapshot:
int32      43
float64    37
int8        1
Name: count, dtype: int64
string[python]    1
Name: count, dtype: int64

Transformed shapes: (436361, 85) (26648, 85) (13324, 85)
✅ Artifacts is consistent.
